# 4. Diet problem (Week 2 tutorial Q6c; the slides call it Q5c): the fix, with limits on fat and vitamins

Lecture: *Week 3 - Linear Programming 2*, PDF pages 14–15 (*Fix* and *5c Solution via Pyomo (Problem Based)*).

**The problem.** Maximise attractiveness $\sum_i a_i x_i$ as in demo 3, but every nutrient now has a minimum and a
maximum (maximum = minimum + 20%):

| | Minimum | Maximum |
| --- | --- | --- |
| Vitamin A | 100 | 120 |
| Vitamin B | 120 | 144 |
| Vitamin C | 200 | 240 |
| Vitamin D | 80 | 96 |
| Fat | 250 | 300 |

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Optimal, attractiveness **149.66**, with `x[0]` = 0.19, `x[5]` = 7.63, `x[6]` = 0.86,
`x[7]` = 5.93, `x[9]` = 7.16 (the same as Week 02's Q6c notebook).

*Original script: `5 Source Files/Python/Week2_question_5c_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

The data: attractiveness, vitamin and fat content per unit of each food; position `i` is food `i + 1`.

In [ ]:
# Define the attractiveness, vitamin, and fat values as lists
a = [6, 1, 5, 5, 6, 4, 10, 10, 6, 7]
vita = [18, 8, 22, 6, 0, 0, 8, 4, 0, 12]
vitb = [4, 6, 10, 12, 14, 0, 0, 0, 6, 20]
vitc = [0, 22, 0, 0, 0, 20, 0, 8, 10, 0]
vitd = [6, 14, 0, 6, 10, 8, 6, 0, 12, 4]
fat = [0, 0, 8, 0, 0, 6, 20, 40, 0, 0]

no_vars = len(a)

A *concrete model* is the container for the LP.

In [ ]:
# Create a concrete model
model = ConcreteModel()

Ten non-negative variables: `x[i]` is the amount of food `i + 1`.

In [ ]:
# Define the decision variables
model.x = Var(range(10), domain=NonNegativeReals)

The objective: total attractiveness, maximised (as in demo 3).

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(a[i] * model.x[i] for i in range(no_vars)), sense=maximize)

The original five constraints from demo 2.

In [ ]:
# Define constraints
model.vita_min = Constraint(expr=sum(vita[i] * model.x[i] for i in range(no_vars)) >= 100)  # vitamin A: at least 100
model.vitb_min = Constraint(expr=sum(vitb[i] * model.x[i] for i in range(no_vars)) >= 120)  # vitamin B: at least 120
model.vitc_min = Constraint(expr=sum(vitc[i] * model.x[i] for i in range(no_vars)) >= 200)  # vitamin C: at least 200
model.vitd_min = Constraint(expr=sum(vitd[i] * model.x[i] for i in range(no_vars)) >= 80)  # vitamin D: at least 80
model.fat_max = Constraint(expr=sum(fat[i] * model.x[i] for i in range(no_vars)) <= 300)  # fat: at most 300

**The fix**: an upper limit on each vitamin (minimum × 1.2) and a lower limit on fat (300 / 1.2 = 250, so that 300 is 250 + 20%). Now every food adds to at least one limited nutrient, so the diet cannot grow forever.

In [ ]:
# Additional constraints: the other end of each range (slide 'Fix')
model.vita_max = Constraint(expr=sum(vita[i] * model.x[i] for i in range(no_vars)) <= 100 * 1.2)  # vitamin A: at most 120
model.vitb_max = Constraint(expr=sum(vitb[i] * model.x[i] for i in range(no_vars)) <= 120 * 1.2)  # vitamin B: at most 144
model.vitc_max = Constraint(expr=sum(vitc[i] * model.x[i] for i in range(no_vars)) <= 200 * 1.2)  # vitamin C: at most 240
model.vitd_max = Constraint(expr=sum(vitd[i] * model.x[i] for i in range(no_vars)) <= 80 * 1.2)  # vitamin D: at most 96
model.fat_min = Constraint(expr=sum(fat[i] * model.x[i] for i in range(no_vars)) >= 300 / 1.2)  # fat: at least 250

Solve with HiGHS; `tee=True` prints the solver's log.

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=True)

If the solver reports *optimal*, print the attractiveness and the amount of each food.

In [ ]:
# Check the solver status and print the results
if results.solver.status == SolverStatus.ok:
    if results.solver.termination_condition == TerminationCondition.optimal:
        print("Optimal solution found.")
        print("Objective value:", model.obj())
        print("Decision variables:")
        for i in range(no_vars):
            print(f"x[{i}] = {model.x[i].value}")
    else:
        print("Solver terminated.")
else:
    print("Solver did not find a solution. Status:", results.solver.status)
    
    

## What do you notice?

Compare with demo 2: which foods does the most attractive diet use, and does it respect the fat limit?

<details><summary>Answer</summary>

It uses foods 1, 6, 7, 8 and 10 (`x[0]`, `x[5]`, `x[6]`, `x[7]`, `x[9]`), including the fatty but very
attractive foods 7 and 8 (attractiveness 10 each), so the fat limits now matter. The cheapest diet in demo 2
used foods 2, 5 and 10 and no fat at all. Different objectives give very different "best" diets from the
same data.
</details>